# Parte 1 - Recolección de noticias
- **Opción elegida:** A (La República)
- **Fecha de descarga:** 18/10/2026
- **URLs base:** https://larepublica.pe/rss/{seccion}.xml, https://larepublica.pe/sitemap/{seccion}.xml, https://larepublica.pe/{seccion}
- **Método:** RSS + sitemaps oficiales como semillas; luego visita de noticias individuales con requests + BeautifulSoup (pausa de 1.5 s, 1 reintento, respeta robots.txt).

## 1. Configuración: librerías y descarga respetuosa
Se revisa `robots.txt`, se usa un User-Agent descriptivo, una pausa de 1.5 s entre solicitudes y como máximo 1 reintento, como piden las buenas prácticas de scraping del enunciado.

In [3]:
import re, json, time
from pathlib import Path
from datetime import datetime
from collections import deque, Counter
from urllib.parse import urljoin, urldefrag
from urllib.robotparser import RobotFileParser

import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://larepublica.pe"
HEADERS = {"User-Agent": "TareaDataSciencePython/1.0 (proyecto académico)"}
RAW = Path("datos/raw"); RAW.mkdir(parents=True, exist_ok=True)

rp = RobotFileParser(BASE + "/robots.txt")
rp.read()

def get(url):
    """Descarga con pausa de 1.5 s y como máximo 1 reintento."""
    if not rp.can_fetch(HEADERS["User-Agent"], url):
        print("robots.txt no lo permite:", url)
        return None
    for intento in range(2):
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            time.sleep(1.5)
            if r.status_code == 200:
                return r.text
            print(r.status_code, url)
        except requests.RequestException as e:
            print("Error:", e)
            time.sleep(1.5)
    return None


## 2. Extracción
### 2.1 Semillas desde los RSS y sitemaps oficiales
Se descargan el RSS y el sitemap de cada sección y se guarda una copia cruda en `datos/raw/`.

In [4]:
# Nombres de los RSS/sitemaps de La República (aparecen listados en su robots.txt)
SECCIONES = ["politica", "economia", "deportes", "ciencia", "tecnologia",  # las que dan las 4 categorías
             "sociedad", "mundo", "espectaculos",                           # se descartan: sirven para el reporte de descartes
             "home"]                                                        # portada: mezcla todas las secciones

registros = []   # una fila por noticia encontrada

for sec in SECCIONES:
    # --- RSS ---
    xml = get(f"{BASE}/rss/{sec}.xml")
    if xml:
        (RAW / f"rss_{sec}.xml").write_text(xml, encoding="utf-8")   # respaldo crudo
        soup = BeautifulSoup(xml, "xml")
        for it in soup.find_all("item"):
            registros.append({
                "titular": it.title.get_text(strip=True) if it.title else None,
                "bajada": it.description.get_text(strip=True) if it.description else None,
                "url": it.link.get_text(strip=True) if it.link else None,
                "fecha": it.pubDate.get_text(strip=True) if it.pubDate else None,
                "fuente": f"rss_{sec}",
            })
    # --- Sitemap de noticias ---
    xml = get(f"{BASE}/sitemap/{sec}.xml")
    if xml:
        (RAW / f"sitemap_{sec}.xml").write_text(xml, encoding="utf-8")
        soup = BeautifulSoup(xml, "xml")
        for u in soup.find_all("url"):
            t = u.find("title")
            f = u.find("publication_date") or u.find("lastmod")
            registros.append({
                "titular": t.get_text(strip=True) if t else None,
                "bajada": None,      # el sitemap no trae bajada; se completa al visitar la noticia
                "url": u.loc.get_text(strip=True),
                "fecha": f.get_text(strip=True) if f else None,
                "fuente": f"sitemap_{sec}",
            })

semillas = pd.DataFrame(registros)
print(semillas.shape)
semillas["fuente"].value_counts()

410 https://larepublica.pe/rss/tecnologia.xml
410 https://larepublica.pe/rss/tecnologia.xml
410 https://larepublica.pe/sitemap/tecnologia.xml
410 https://larepublica.pe/sitemap/tecnologia.xml
404 https://larepublica.pe/sitemap/home.xml
404 https://larepublica.pe/sitemap/home.xml
(504, 5)


fuente
rss_home                200
rss_sociedad             46
sitemap_sociedad         46
rss_espectaculos         31
sitemap_espectaculos     31
rss_mundo                23
sitemap_mundo            23
rss_politica             22
sitemap_politica         22
rss_deportes             14
sitemap_deportes         14
rss_ciencia              10
sitemap_ciencia          10
rss_economia              6
sitemap_economia          6
Name: count, dtype: int64

### 2.2 Sección original a partir de la URL
La sección (`seccion_original`) y la fecha se leen de la propia URL de cada noticia.

In [5]:
PATRON_NOTICIA = re.compile(r"^https://larepublica\.pe/([\w-]+)/(?:[\w-]+/)?(\d{4})/(\d{2})/(\d{2})/[\w-]+$")

def seccion_de(url):
    m = PATRON_NOTICIA.match(url or "")
    return m.group(1) if m else None

semillas["seccion_original"] = semillas["url"].map(seccion_de)
semillas["seccion_original"].value_counts()

seccion_original
sociedad            134
espectaculos         93
mundo                69
politica             68
deportes             42
ciencia              30
economia             18
opinion              13
latinos-en-usa        8
cultural              5
cine-series           5
entretenimiento       4
verificador           4
mastergrama           2
carlincatura          2
horoscopo             2
cultura-asiatica      2
servicios             2
heduardicidios        1
Name: count, dtype: int64

### 2.3 Ampliación visitando noticias
Los RSS y sitemaps no alcanzan las ~100 noticias por categoría, así que se visitan las noticias y se siguen sus enlaces internos hasta completar la cuota. Cada noticia descargada se respalda en `datos/raw/articulos_raw.jsonl`.

In [ ]:
# Secciones que cuentan para cada categoría
MAPEO = {
    "politica": "politica",
    "economia": "economia",
    "deportes": "deportes",
    "ciencia": "ciencia_tecnologia",
    "tecnologia": "ciencia_tecnologia",
}
CUOTA = 110          # algo más de 100 por categoría, para tener margen tras limpiar
MAX_PAGINAS = 800    # tope de seguridad (~20 min con pausas de 1.5 s)

def parsear_noticia(url, html):
    s = BeautifulSoup(html, "html.parser")
    def meta(**attrs):
        tag = s.find("meta", attrs=attrs)
        return tag.get("content") if tag else None
    titular = s.h1.get_text(strip=True) if s.h1 else meta(property="og:title")
    bajada = meta(property="og:description") or meta(name="description")
    fecha = meta(property="article:published_time")
    if not fecha:                                   # respaldo: fecha de la URL
        m = PATRON_NOTICIA.match(url)
        fecha = f"{m.group(2)}-{m.group(3)}-{m.group(4)}" if m else None
    enlaces = set()
    for a in s.find_all("a", href=True):
        u = urldefrag(urljoin(BASE, a["href"]))[0]
        if PATRON_NOTICIA.match(u):
            enlaces.add(u)
    return {"titular": titular, "bajada": bajada, "url": url, "fecha": fecha,
            "seccion_original": seccion_de(url), "fuente": "articulo"}, enlaces

conteo = Counter()
visitadas, cola = set(), deque()

# Semillas: URLs de RSS/sitemaps + enlaces de las páginas de sección
for u in semillas["url"].dropna().unique():
    cola.append(u)
for sec in MAPEO:
    html = get(f"{BASE}/{sec}")
    if html:
        _, enlaces = parsear_noticia(f"{BASE}/{sec}", html)
        cola.extend(enlaces)

articulos = []
with open(RAW / "articulos_raw.jsonl", "a", encoding="utf-8") as respaldo:
    while cola and len(visitadas) < MAX_PAGINAS:
        url = cola.popleft()
        cat = MAPEO.get(seccion_de(url))
        if url in visitadas or cat is None or conteo[cat] >= CUOTA:
            continue
        visitadas.add(url)
        html = get(url)
        if not html:
            continue
        fila, enlaces = parsear_noticia(url, html)
        articulos.append(fila)
        respaldo.write(json.dumps(fila, ensure_ascii=False) + "\n")   # respaldo crudo
        conteo[cat] += 1
        # prioriza enlaces de categorías que aún no llegan a la cuota
        for e in enlaces:
            c = MAPEO.get(seccion_de(e))
            if e not in visitadas and c and conteo[c] < CUOTA:
                cola.append(e)
        if len(visitadas) % 25 == 0:
            print(len(visitadas), dict(conteo))
        if all(conteo[c] >= CUOTA for c in set(MAPEO.values())):
            break

print("Final:", dict(conteo))

25 {'politica': 22, 'economia': 3}
50 {'politica': 22, 'economia': 6, 'deportes': 14, 'ciencia_tecnologia': 8}
75 {'politica': 26, 'economia': 13, 'deportes': 14, 'ciencia_tecnologia': 22}
100 {'politica': 51, 'economia': 13, 'deportes': 14, 'ciencia_tecnologia': 22}
125 {'politica': 67, 'economia': 22, 'deportes': 14, 'ciencia_tecnologia': 22}
150 {'politica': 68, 'economia': 46, 'deportes': 14, 'ciencia_tecnologia': 22}
175 {'politica': 68, 'economia': 48, 'deportes': 37, 'ciencia_tecnologia': 22}
404 https://larepublica.pe/deportes/2026/10/12/franco-velazco-critico-contra-pablo-guede-y-miguel-rondelli-tras-polemica-con-universitario-la-solidaridad-la-vamos-a-ver-a-fin-de-ano-833664
404 https://larepublica.pe/deportes/2026/10/12/franco-velazco-critico-contra-pablo-guede-y-miguel-rondelli-tras-polemica-con-universitario-la-solidaridad-la-vamos-a-ver-a-fin-de-ano-833664
200 {'politica': 68, 'economia': 48, 'deportes': 59, 'ciencia_tecnologia': 24}
225 {'politica': 68, 'economia': 48, '

## 3. Limpieza y normalización
Se eliminan duplicados por URL y titulares vacíos, se mapea `seccion_original` a las 4 categorías oficiales y se reporta cuántas noticias se descartaron y por qué.

In [7]:
df = pd.concat([pd.DataFrame(articulos), semillas], ignore_index=True)
n_inicial = len(df)
descartes = {}

# 1) Preferir la versión visitada (tiene bajada) y eliminar duplicados por URL
df["prioridad"] = (df["fuente"] == "articulo").astype(int)
df = df.sort_values("prioridad", ascending=False)
antes = len(df); df = df.drop_duplicates("url"); descartes["duplicados por URL"] = antes - len(df)

# 2) Titulares vacíos
df["titular"] = df["titular"].fillna("").str.strip()
antes = len(df); df = df[df["titular"] != ""]; descartes["titular vacío"] = antes - len(df)

# 3) Mapeo a las 4 categorías
df["seccion_original"] = df["url"].map(seccion_de)
df["etiqueta"] = df["seccion_original"].map(MAPEO)
fuera = df[df["etiqueta"].isna()]
descartes["sección fuera de las 4 categorías"] = len(fuera)
print("Secciones descartadas:", fuera["seccion_original"].value_counts().to_dict())
df = df[df["etiqueta"].notna()]

# 4) Balancear: como máximo 100 por categoría (las más recientes)
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce", utc=True)
df = df.sort_values("fecha", ascending=False)
antes = len(df); df = df.groupby("etiqueta").head(100); descartes["exceso sobre 100 por categoría"] = antes - len(df)

df["medio"] = "La República"
df["bajada"] = df["bajada"].fillna("")
df["fecha"] = df["fecha"].dt.strftime("%Y-%m-%d")

print(f"Inicial: {n_inicial} | Final: {len(df)}")
pd.Series(descartes, name="filas descartadas").to_frame()

Secciones descartadas: {'sociedad': 45, 'espectaculos': 31, 'mundo': 23, 'opinion': 13, 'latinos-en-usa': 8, 'cultural': 5, 'cine-series': 5, 'entretenimiento': 4, 'verificador': 4, 'carlincatura': 2, 'servicios': 2, 'cultura-asiatica': 2, 'horoscopo': 2, 'mastergrama': 2, 'heduardicidios': 1}
Inicial: 944 | Final: 400


,filas descartadas
duplicados por URL,355
titular vacío,0
sección fuera de las 4 categorías,149
exceso sobre 100 por categoría,40


### 3.1 Tabla de equivalencias: sección → categoría

In [8]:
pd.DataFrame(sorted(MAPEO.items()), columns=["seccion_original", "etiqueta"])

,seccion_original,etiqueta
0,ciencia,ciencia_tecnologia
1,deportes,deportes
2,economia,economia
3,politica,politica
4,tecnologia,ciencia_tecnologia


### 3.2 Noticias descartadas
Se partió de 944 filas (440 noticias visitadas + 504 entradas de RSS y sitemaps) y quedaron **400 noticias, 100 por categoría**:

- **355 duplicados por URL:** la misma noticia aparecía en el RSS, en el sitemap y/o fue visitada. Se conservó la versión visitada porque incluye la bajada.
- **0 titulares vacíos.**
- **149 noticias de secciones que no encajan en las 4 categorías:** sociedad (45), espectáculos (31), mundo (23), opinión (13) y otras 37 de secciones menores (latinos-en-usa, cultural, cine-series, entretenimiento, verificador, horóscopo, etc.). Se descartaron porque el enunciado no permite crear una quinta categoría.
- **40 por exceso:** se descargaron 110 por categoría como margen y se conservaron las 100 más recientes de cada una, para tener un dataset balanceado.